# RADCLIM: period of the copy, one file, one hour on the OPERA grid

S5 (#12). RADCLIM (RMI) is radar rainfall for Belgium merged with about 150 rain gauges; it is a candidate
bridge for H3 and the accumulation reference for H5. The copy used here is the one on radar.flood4cast.be
(product `kmi-radclim`, behind a login), downloaded by hand as a zip per period into
`DATA_DIR / "radclim"`; the zip keeps the archive layout `kmi-radclim/YYYY/MM/<YYYYmmddHHMMSS>.rad.best.comp.rate.qpe_edk.hdf`.

Source for the product: the RADCLIM user guide, version 2
(`docs/sources/2023 - RMI - RADCLIM user guide.md`).

1. Period and gaps of the copy, as the download page reports them.
2. One 5 min file: structure, projection, grid, units, time attributes.
3. The rainiest hour of 14/7/2021 over Belgium, brought onto the OPERA grid and set beside the ODYSSEY
   1 h accumulation (`ACRR_QIND`) of the same window.

In [ ]:
from opera_ds.paths import DATA_DIR

import datetime as dt
import json
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from pyproj import Transformer

from opera_ds.odim_io import read_odim_composite, refine_where
from opera_ds.ord import download

sys.path.insert(0, str(Path.cwd().parent))          # scripts/ is not a package of opera_ds
from scripts.make_region_mask import country_geometry, grid_mask

## Parameters

In [ ]:
RADCLIM_DIR = DATA_DIR / "radclim"
DAY = dt.date(2021, 7, 14)        # floods of July 2021; the user guide is published with that RMI case
OPERA_DIR = DATA_DIR / "odyssey_acrr"


def radclim_path(t):
    return RADCLIM_DIR / "kmi-radclim" / f"{t:%Y}" / f"{t:%m}" / f"{t:%Y%m%d%H%M%S}.rad.best.comp.rate.qpe_edk.hdf"

## 1. Period and gaps of the copy

From the download page (step 3, *What is there*, read on 9/11/2026), not recomputed here: the whole archive
is not downloaded. Archive 2017-01-01 00:05 → 2023-11-30 23:55 UTC, 726,325 files. At one file every 5 min
the period has 727,199 time steps, so 874 are missing (0.12%).

In [ ]:
a, b = dt.datetime(2017, 1, 1, 0, 5), dt.datetime(2023, 11, 30, 23, 55)
expected = int((b - a).total_seconds() // 300) + 1
print(f"expected {expected:,}, in the archive 726,325, missing {expected - 726_325:,} ({(expected - 726_325) / expected:.2%})")

manifest = json.loads((RADCLIM_DIR / "manifest.json").read_text(encoding="utf-8"))
files = [RADCLIM_DIR / f["file"] for f in manifest["files"]]
print(manifest["start"], "->", manifest["end"], f"{len(files)} files in the zip, all present:", all(p.exists() for p in files))
print(f"{sum(f['size'] for f in manifest['files']) / len(files) / 1e3:.0f} kB per file on this day")

## 2. One 5 min file

Every group and attribute of the first file of the day, then the reader of the project
(`read_odim_composite`, quantity `RATE`) on the same file.

In [ ]:
def attrs(g):
    return {k: (v.decode() if isinstance(v, bytes) else v) for k, v in g.attrs.items()}


p0 = radclim_path(dt.datetime.combine(DAY, dt.time(0, 0)))
with h5py.File(p0, "r") as f:
    print("root", attrs(f))
    f.visititems(lambda n, o: print(n, getattr(o, "shape", ""), getattr(o, "dtype", ""), attrs(o) if o.attrs else ""))

fr = read_odim_composite(str(p0), "RATE")
print("\nshape", fr.values.shape, "| covered", f"{fr.coverage.mean():.1%}", "| time", fr.time)
print("grid", {k: fr.where[k] for k in ("UL_x", "UL_y", "xscale", "yscale", "xsize", "ysize")})

In [ ]:
# the projection is Belgian Lambert 2008 (EPSG:3812): same transformation as the projdef of the file
t_file = Transformer.from_crs("EPSG:4326", fr.where["projdef"], always_xy=True)
t_3812 = Transformer.from_crs("EPSG:4326", "EPSG:3812", always_xy=True)
for lon, lat in [(4.36, 50.80), (2.5, 51.1), (6.4, 49.5)]:
    print((lon, lat), np.round(t_file.transform(lon, lat), 2), np.round(t_3812.transform(lon, lat), 2))

# time attributes of all files: label against the window declared in dataset1/what
lag = []
for p in files:
    with h5py.File(p, "r") as f:
        r, w = attrs(f["what"]), attrs(f["dataset1/what"])
    t = dt.datetime.strptime(r["date"] + r["time"], "%Y%m%d%H%M%S")
    s = dt.datetime.strptime(w["startdate"] + w["starttime"], "%Y%m%d%H%M%S")
    e = dt.datetime.strptime(w["enddate"] + w["endtime"], "%Y%m%d%H%M%S")
    lag.append(((t - s).total_seconds(), (t - e).total_seconds()))
lag = np.array(lag)
print("label minus start: %d-%d s, label minus end: %d-%d s" % (lag[:, 0].min(), lag[:, 0].max(), lag[:, 1].min(), lag[:, 1].max()))

v = np.stack([read_odim_composite(str(p), "RATE").values for p in files])
print(f"values: min {np.nanmin(v):.2f}, max {np.nanmax(v):.1f} (mm/h per the download page)")

## 3. One hour on the OPERA grid

**Window.** ODYSSEY `ACRR` labelled H covers H−55 → H+5 (`docs/data.md`, *Time windows*; the file declares
it too). RADCLIM values end at their label (user guide, *Temporal resolution*: "the timestamp corresponds to
the end of the accumulation period"), so the frames labelled H−50 … H+5 cover the same 60 min:
12 frames × 5/60 h, values in mm/h.

**Hour.** The hour H of 14/7/2021 with the highest mean RADCLIM accumulation over Belgium (Natural Earth
polygon, pixel centres inside).

**Grid.** RADCLIM is put on the OPERA 1 km grid (the ODYSSEY 2 km grid refined by 2, the CIRRUS grid) by
nearest neighbour: both grids are 1 km, so each OPERA pixel takes the RADCLIM pixel that contains its
centre, without changing values. The 1 km field is then averaged on 2 × 2 blocks to meet ODYSSEY at 2 km
(the user guide: each RADCLIM value is the mean precipitation over its 1 km square).

In [ ]:
def radclim_hour(H):
    frames = [read_odim_composite(str(radclim_path(H + dt.timedelta(minutes=m))), "RATE") for m in range(-50, 10, 5)]
    acc = np.sum([f.values for f in frames], axis=0) * 5 / 60
    cov = np.all([f.coverage for f in frames], axis=0)
    return np.where(cov, acc, np.nan), frames[0].where


_, rw = radclim_hour(dt.datetime.combine(DAY, dt.time(1)))
bel_geom = country_geometry("BEL", str(DATA_DIR / "naturalearth"))
to_rad = Transformer.from_crs("EPSG:4326", rw["projdef"], always_xy=True)
from shapely import contains_xy, transform
g_rad = transform(bel_geom, lambda c: np.column_stack(to_rad.transform(c[:, 0], c[:, 1])))
xr = rw["UL_x"] + (np.arange(rw["xsize"]) + 0.5) * rw["xscale"]
yr = rw["UL_y"] - (np.arange(rw["ysize"]) + 0.5) * rw["yscale"]
bel_rad = contains_xy(g_rad, *np.meshgrid(xr, yr))

hours = [dt.datetime.combine(DAY, dt.time(h)) for h in range(1, 24)]      # H+5 must be in the day's zip
means = {H: np.nanmean(radclim_hour(H)[0][bel_rad]) for H in hours}
H = max(means, key=means.get)
for h, m in means.items():
    print(f"{h:%H:%M}  {m:5.2f} mm" + ("  <- rainiest" if h == H else ""))

In [ ]:
rad, _ = radclim_hour(H)
acrr = read_odim_composite(download(H, "ACRR_QIND", str(OPERA_DIR)), "ACRR")
with h5py.File(acrr.path, "r") as f:
    w = attrs(f["dataset1/what"])
print("ODYSSEY ACRR window:", w["startdate"], w["starttime"], "->", w["enddate"], w["endtime"])

# OPERA 1 km grid around the RADCLIM domain
w1 = refine_where(acrr.where, 2)
to_laea = Transformer.from_crs("EPSG:4326", w1["projdef"], always_xy=True)
x0, y0 = to_laea.transform(float(w1["UL_lon"]), float(w1["UL_lat"]))
corners = [to_rad.transform(lo, la, direction="INVERSE") for lo, la in
           [(rw["UL_x"], rw["UL_y"]), (rw["UL_x"] + 700e3, rw["UL_y"]), (rw["UL_x"], rw["UL_y"] - 700e3), (rw["UL_x"] + 700e3, rw["UL_y"] - 700e3)]]
cx, cy = to_laea.transform(*zip(*corners))
c0, c1 = int((min(cx) - x0) // 2000) * 2, int((max(cx) - x0) // 2000 + 1) * 2     # even, so 2 x 2 blocks align with ODYSSEY
r0, r1 = int((y0 - max(cy)) // 2000) * 2, int((y0 - min(cy)) // 2000 + 1) * 2
xs = x0 + (np.arange(c0, c1) + 0.5) * 1000.0
ys = y0 - (np.arange(r0, r1) + 0.5) * 1000.0
X, Y = np.meshgrid(xs, ys)
lam = Transformer.from_crs(w1["projdef"], rw["projdef"], always_xy=True)
XR, YR = lam.transform(X, Y)
ci = np.floor((XR - rw["UL_x"]) / rw["xscale"]).astype(int)
ri = np.floor((rw["UL_y"] - YR) / rw["yscale"]).astype(int)
inside = (ci >= 0) & (ci < rw["xsize"]) & (ri >= 0) & (ri < rw["ysize"])
rad1 = np.full(X.shape, np.nan)
rad1[inside] = rad[ri[inside], ci[inside]]

ny, nx = rad1.shape
rad2 = rad1.reshape(ny // 2, 2, nx // 2, 2).mean(axis=(1, 3))          # NaN if any 1 km pixel is NaN
od2 = np.where(acrr.coverage, acrr.values, np.nan)[r0 // 2:r1 // 2, c0 // 2:c1 // 2]
bel2 = grid_mask(acrr.where, bel_geom)[r0 // 2:r1 // 2, c0 // 2:c1 // 2]
both = np.isfinite(rad2) & np.isfinite(od2)
print(f"OPERA 1 km window rows {r0}-{r1}, cols {c0}-{c1}; RADCLIM on {np.isfinite(rad1).mean():.0%} of it")

In [ ]:
def stats(m, name):
    a, o = rad2[m & both], od2[m & both]
    print(f"{name:20s} pixels {a.size:6d}  mean RADCLIM {a.mean():5.2f} mm  ODYSSEY {o.mean():5.2f} mm  "
          f"ratio {o.mean() / a.mean():4.2f}  corr {np.corrcoef(a, o)[0, 1]:4.2f}  max {a.max():5.1f} / {o.max():5.1f} mm")


stats(bel2, "Belgium")
stats(~bel2, "rest of the domain")

fig, ax = plt.subplots(1, 4, figsize=(20, 5.5), layout="constrained")
kw = dict(cmap="viridis", vmin=0, vmax=20)
ax[0].imshow(rad1, **kw); ax[0].set_title(f"RADCLIM on the OPERA 1 km grid\n{H - dt.timedelta(minutes=55):%d/%m %H:%M}–{H + dt.timedelta(minutes=5):%H:%M} UTC")
ax[1].imshow(rad2, **kw); ax[1].set_title("RADCLIM, 2 × 2 mean (2 km)")
im = ax[2].imshow(od2, **kw); ax[2].set_title(f"ODYSSEY ACRR_QIND {H:%H:%M} (2 km)")
d = ax[3].imshow(np.where(both, od2 - rad2, np.nan), cmap="RdBu", vmin=-10, vmax=10); ax[3].set_title("ODYSSEY − RADCLIM (2 km)")
for i, a_ in enumerate(ax):
    a_.contour(np.kron(bel2, np.ones((2, 2))) if i == 0 else bel2, levels=[0.5], colors="white" if i < 3 else "k", linewidths=0.8)
    a_.set_xticks([]); a_.set_yticks([])
fig.colorbar(im, ax=ax[2], shrink=0.8, label="mm in 1 h"); fig.colorbar(d, ax=ax[3], shrink=0.8, label="mm")
FIG = Path.cwd().parent / "docs" / "figures" / "radclim" / f"radclim-odyssey-{H:%Y%m%dT%H%M}.png"
FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=80)
print(f"saved {FIG.name}, {FIG.stat().st_size / 1e3:.0f} kB")
plt.show()

## 4. What is seen

Run on 9/11/2026, one hour (14/7/2021, 20:05–21:05 UTC), so nothing here is general.

- **The copy is as described**, except for the content: 700 × 700 at 1 km in EPSG:3812 (the `projdef` of
  the files gives the same coordinates), the four radars of the user guide in `what/source`, but only 5 min
  `RATE` files, no 1 h accumulation. Each file declares a 20 s window ending about 33 s before its label.
- **Same pattern, different level.** Over Belgium the two fields correlate at 0.89 on 2 km pixels, but
  ODYSSEY gives about half of RADCLIM (mean 0.99 against 1.88 mm, maximum 10 against 25 mm); over the rest of
  the RADCLIM domain the ratio is 0.82. The level difference between ODYSSEY and gauge-adjusted references
  belongs to H3.
- **Coverage.** Within the radar circles RADCLIM is NaN in a patch in the north-west of the domain and
  along two thin radial lines, west and south-east of Belgium; what they are is not checked. Inside Belgium
  every pixel has a value in this hour.
- **No overlap with CIRRUS.** The copy ends on 30/11/2023 and CIRRUS starts on 1/7/2024, so RADCLIM can be
  compared with ODYSSEY only.